# Tensors

In [ ]:
import numpy
import torch

shape = (2, 3,)
rand_tensor = torch.rand(shape)                                     # random
zeros_tensor = torch.tensor(numpy.zeros(shape), dtype=torch.int32)  # zeros from numpy
sequ_tensor = torch.tensor([1, 2, 3, 4, 5])                         # from list

print(rand_tensor)
print(zeros_tensor)
print(sequ_tensor)

## Eigenschaften von Tensoren

Cast `d-type`

In [ ]:
zeros_tensor.type(torch.float16)

## Indexing

In [ ]:
print(sequ_tensor[2].item())
print(sequ_tensor[2:4])
print(sequ_tensor[-1])

## Operationen

In [ ]:
t1 = torch.tensor([[1, 2],[3,4]])
t2 = torch.tensor([[5,6],[7,8]])

print(f"elementwise multiplication: \n{t1 * t2}")
print(f"matrix multiplication: \n{t1 @ t2}")
print(f"concat at dim 0: \n{torch.concat((t1, t2),dim=0)}")
print(f"concat at dim 1: \n{torch.concat((t1, t2),dim=1)}")

In [ ]:
t = torch.tensor([-1,-2])
t.abs_()
print(f"inplace operation {t}")

In [ ]:
x = torch.randn(2, 3, 5)
print(x.size())
print(torch.permute(x, (2, 0, 1)).size())

In [ ]:
x = torch.zeros(2, 1, 2, 1, 2)
print(x.size())
y = torch.squeeze(x)
print(y.size())
y = torch.squeeze(x, 0)
print(y.size())
y = torch.squeeze(x, 1)
print(y.size())
y = torch.squeeze(x, (1, 2, 3))
print(y.size())

# Beispiel IMDB

In [ ]:
import datasets
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.optim import AdamW
from torch.utils.data import DataLoader
import numpy as np

# Parameter
FEATURE_SIZE = 10000
NUM_CLASSES = 2
BATCH_SIZE = 16

# Data

train, test = datasets.load_dataset("imdb", split=["train", "test"])
test = test.train_test_split()['test']

train[0]


In [ ]:
train, test = datasets.load_dataset("imdb", split=["train[35%:65%]", "test[45%:55%]"]) # 10 % der Testdaten reichen < 50%->label =0; > 50%->label =1 

tfidf = TfidfVectorizer(stop_words="english", max_features=FEATURE_SIZE, dtype=np.float32)
x = [sample["text"] for sample in train]
tfidf.fit(x)    # Term Frequencies und Document Frequencies zählen

In [ ]:
def transform(example):
    features = tfidf.transform([example['text']])
    # return features.toarray()
    return {"features": features.toarray()}

train = train.map(transform)
test = test.map(transform)
# Falls Dict in Dataloader, batch dim is 1 -> konvertieren in list(features, label)
train = [(torch.tensor(sample["features"]), torch.nn.functional.one_hot(torch.tensor(sample['label']), num_classes=NUM_CLASSES)) for sample in train]
test = [(torch.tensor(sample["features"]), torch.nn.functional.one_hot(torch.tensor(sample['label']), num_classes=NUM_CLASSES)) for sample in test]
train_dataloader = DataLoader(train, shuffle=True, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test, shuffle=False, batch_size=BATCH_SIZE)
train[0]

In [ ]:
# Model
# model = torch.nn.Linear(in_features=FEATURE_SIZE, out_features=NUM_CLASSES)
# model = torch.nn.Sequential( torch.nn.Linear(in_features=FEATURE_SIZE, out_features=NUM_CLASSES), torch.nn.Tanh())
model = torch.nn.Sequential( torch.nn.Linear(in_features=FEATURE_SIZE, out_features=500), 
                              torch.nn.Tanh(),
                              torch.nn.Linear(in_features=500, out_features=NUM_CLASSES), 
                              torch.nn.Tanh())
learning_rate = 1e-5
optim = AdamW(model.parameters(), lr=learning_rate)
criteria = torch.nn.CrossEntropyLoss()

 ## Trainingsfunktionen

In [ ]:

def train_step(input, target):
    output = model(input)
    loss = criteria(output.squeeze(), target.float())
    loss.backward()
    optim.step()
    optim.zero_grad()


def eval_step(input, target):
    output = model(input).squeeze()
    loss = criteria(output, target.float()).item()
    return float(loss), (output.argmax(1) == target.argmax(1)).type(torch.float).sum().item()


def evaluate():
    model.eval()
    loss_global = 0
    corrects = 0

    sample_size = len(test_dataloader) * BATCH_SIZE
    test_iter = iter(test_dataloader)
    with torch.no_grad():
        for features, label in test_iter:
            loss, correct = eval_step(features, label)
            loss_global += loss
            corrects += correct
    return loss_global / sample_size, corrects / sample_size

In [ ]:
num_epochs = 10

for e in range(num_epochs):
    model.train()
    train_iter = iter(train_dataloader)
    for input,target in train_iter:
        train_step(input, target)

    loss, accuracy = evaluate()
    print(f"Epoch = {e}, loss = {loss:0.4f}, accuracy = {accuracy:0.4f}")

In [ ]:
features, label = test[0]
print(model(features))
print(label)
